In [16]:
import pandas as pd
import numpy as np
from itertools import product
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import GroupKFold, RandomizedSearchCV, GridSearchCV
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    AdaBoostRegressor,
)
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
    r2_score,
)
from sklearn.base import clone
from IPython.display import display

pd.set_option("display.max_columns", None)

from methods.ml_models import prepare_features, make_search
from data_preparation import NON_TEMPORAL_COLUMNS

## Pre-processing

In [2]:
data = pd.read_parquet("data/full_processed.parquet")

In [3]:
data.head()

,id,time,tad,m155,age,sex,wgt,hgt,dty,gf,gfc,ifc,dage,cisq,dyl,lym,oifc,cmv,bkv,dm,amt_tac,dv_tac,amt_map,dv_map,dv
0,1,0.00,0.00,0.00,35.0,0.0,81.0,170.0,1.0,6.0,1.0,0.0,32.0,NaN,17.4,31.2,0.0,0.0,0.0,0.0,8.0,0.000000,738.96,NaN,NaN
1,1,142.80,NaN,NaN,35.0,0.0,NaN,170.0,1.0,NaN,NaN,NaN,32.0,NaN,17.4,NaN,NaN,NaN,NaN,0.0,0.0,2.407846,NaN,NaN,NaN
2,1,143.95,11.95,0.08,35.0,0.0,85.0,170.0,1.0,32.0,3.0,0.0,32.0,NaN,17.4,28.1,0.0,0.0,0.0,0.0,NaN,NaN,0.00,3.88,NaN
3,1,144.00,0.00,0.00,35.0,0.0,85.0,170.0,1.0,32.0,3.0,0.0,32.0,NaN,17.4,28.1,0.0,0.0,0.0,0.0,8.0,0.000000,738.96,NaN,NaN
4,1,145.00,1.00,0.00,35.0,0.0,85.0,170.0,1.0,32.0,3.0,0.0,32.0,NaN,17.4,28.1,0.0,0.0,0.0,0.0,0.0,2.698000,0.00,3.29,NaN


In [43]:
mask = pd.notna(data.lym)
clean_lym_sr = data[mask][["id", "time", "lym"]]

mask = pd.notna(data.dv_tac)
clean_input = data[mask][["id", "time", "dv_tac"]]
delta_time = []
tendency = []
value = []

for _, lym in clean_lym_sr.iterrows():
    elegible = clean_input[clean_input.time <= lym.time].iloc[-2:]
    delta_time.append(lym.time - elegible.time.iloc[1])
    value.append(elegible["dv_tac"].iloc[1])
    tendency.append(
        (elegible["dv_tac"].iloc[1] - elegible["dv_tac"].iloc[0])
        / (elegible.time.iloc[1] - elegible.time.iloc[0])
        if elegible.time.iloc[1] - elegible.time.iloc[0] != 0
        else 0
    )

In [19]:
clean_lym_sr.values


array([[1.00000e+00, 0.00000e+00, 3.12000e+01],
       [1.00000e+00, 1.43950e+02, 2.81000e+01],
       [1.00000e+00, 1.44000e+02, 2.81000e+01],
       [1.00000e+00, 1.45000e+02, 2.81000e+01],
       [1.00000e+00, 1.45500e+02, 2.81000e+01],
       [1.00000e+00, 1.46000e+02, 2.81000e+01],
       [1.00000e+00, 1.47000e+02, 2.81000e+01],
       [1.00000e+00, 1.48000e+02, 2.81000e+01],
       [1.00000e+00, 1.50000e+02, 2.81000e+01],
       [1.00000e+00, 1.52000e+02, 2.81000e+01],
       [1.00000e+00, 1.56000e+02, 2.81000e+01],
       [1.00000e+00, 6.47950e+02, 3.55000e+01],
       [2.00000e+00, 0.00000e+00, 1.94000e+01],
       [2.00000e+00, 9.59500e+01, 1.50000e+01],
       [2.00000e+00, 9.60000e+01, 1.50000e+01],
       [2.00000e+00, 9.65000e+01, 1.50000e+01],
       [2.00000e+00, 9.70000e+01, 1.50000e+01],
       [2.00000e+00, 9.75000e+01, 1.50000e+01],
       [2.00000e+00, 9.80000e+01, 1.50000e+01],
       [2.00000e+00, 9.90000e+01, 1.50000e+01],
       [2.00000e+00, 1.00000e+02, 1.5000

## Grid-search

### Random Forest

In [ ]:
cv = GroupKFold(n_splits=5)

rf = RandomForestRegressor(random_state=42, n_jobs=-1)

param_grid = {
    "n_estimators": [100, 200, 300, 500, 800],
    "max_depth": [None, 5, 10, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2", 0.5, 1.0],
}

search_rf = make_search(
    rf, param_grid, search="random", folds=5, n_iter=30,
    seed=42, n_jobs=4, normalization="time-series",
)  # Use search="grid" para Grid Search; ambas maximizam R².

search_rf.fit(X, y, groups=meta_data["id"])

print("Melhores parâmetros:")
display(search_rf.best_params_)

print("Melhor R² médio:")
print(search_rf.best_score_)

### Gradient Boosting

In [ ]:

gb = GradientBoostingRegressor(
    random_state=42
)

param_grid_gb = {
    "n_estimators": [50, 100, 200, 300],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "max_depth": [2, 3, 4, 5],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "subsample": [0.7, 0.8, 1.0],
}

search_gb = make_search(
    gb, param_grid_gb, search="random", folds=5, n_iter=30,
    seed=42, n_jobs=4, normalization="time-series",
)  # Use search="grid" para Grid Search; ambas maximizam R².

search_gb.fit(
    X,
    y,
    groups=meta_data["id"]
)

display(search_gb.best_params_)
print("Melhor R²:", search_gb.best_score_)

### Ada Boost

In [ ]:
ada = AdaBoostRegressor(
    random_state=42
)

param_grid_ada = {
    "n_estimators": [50, 100, 200, 300],
    "learning_rate": [0.01, 0.05, 0.1, 0.5, 1.0],
    "loss": ["linear", "square", "exponential"],
}

search_ada = make_search(
    ada, param_grid_ada, search="random", folds=5, n_iter=20,
    seed=42, n_jobs=4, normalization="time-series",
)  # Use search="grid" para Grid Search; ambas maximizam R².

search_ada.fit(
    X,
    y,
    groups=meta_data["id"]
)

display(search_ada.best_params_)
print("Melhor R²:", search_ada.best_score_)

## Testing

In [ ]:
best_models = {
    "Random Forest": search_rf.best_estimator_,
    "Gradient Boosting": search_gb.best_estimator_,
    "AdaBoost": search_ada.best_estimator_,
}

cv = GroupKFold(n_splits=5)

groups = meta_data["id"]

resultados = []

predicoes = []

for nome_modelo, modelo_base in best_models.items():

    print(f"\nAvaliando {nome_modelo}...")

    for fold, (train_idx, test_idx) in enumerate(
        cv.split(X, y, groups=groups),
        start=1,
    ):

        X_train = X.iloc[train_idx]
        X_test = X.iloc[test_idx]

        y_train = y.iloc[train_idx]
        y_test = y.iloc[test_idx]

        meta_test = meta_data.iloc[test_idx]

        # Cria uma cópia independente do modelo
        modelo = clone(modelo_base)

        # Treino
        modelo.fit(X_train, y_train)

        # Predição
        y_pred = modelo.predict(X_test)

        # Métricas
        mae = mean_absolute_error(y_test, y_pred)

        mape = mean_absolute_percentage_error(
            y_test,
            y_pred,
        )

        rmse = np.sqrt(
            mean_squared_error(
                y_test,
                y_pred,
            )
        )

        r2 = r2_score(
            y_test,
            y_pred,
        )

        resultados.append({
            "modelo": nome_modelo,
            "fold": fold,
            "mae": mae,
            "mape": mape,
            "rmse": rmse,
            "r2": r2,
        })

        # Guardando predições para gráficos posteriores
        fold_pred = meta_test.copy()

        fold_pred["y_true"] = y_test.to_numpy()
        fold_pred["y_pred"] = y_pred
        fold_pred["modelo"] = nome_modelo
        fold_pred["fold"] = fold

        predicoes.append(fold_pred)

In [ ]:
resultados_df = pd.DataFrame(resultados)

predicoes_df = pd.concat(
    predicoes,
    ignore_index=True
)

display(
    resultados_df.style
    .format({
        "mae": "{:.3f}",
        "mape": "{:.3f}",
        "rmse": "{:.3f}",
        "r2": "{:.3f}",
    })
)

In [ ]:
resumo = (
    resultados_df
    .groupby("modelo")[["mae", "mape", "rmse", "r2"]]
    .agg(["mean", "std"])
)

display(resumo.round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

resultados_df.boxplot(
    column="mae",
    by="modelo",
    ax=ax,
)

ax.set_title("MAE por modelo")
ax.set_xlabel("Modelo")
ax.set_ylabel("MAE")

plt.suptitle("")
plt.xticks(rotation=15)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

resultados_df.boxplot(
    column="rmse",
    by="modelo",
    ax=ax,
)

ax.set_title("RMSE por modelo")
ax.set_xlabel("Modelo")
ax.set_ylabel("RMSE")

plt.suptitle("")
plt.xticks(rotation=15)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

resultados_df.boxplot(
    column="r2",
    by="modelo",
    ax=ax,
)

ax.axhline(
    0,
    linestyle="--",
)

ax.set_title("R² por modelo")
ax.set_xlabel("Modelo")
ax.set_ylabel("R²")

plt.suptitle("")
plt.xticks(rotation=15)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

resultados_df.boxplot(
    column="mape",
    by="modelo",
    ax=ax,
)

ax.set_title("MAPE por modelo")
ax.set_xlabel("Modelo")
ax.set_ylabel("MAPE")

plt.suptitle("")
plt.xticks(rotation=15)
plt.show()

In [ ]:
for nome_modelo in best_models:

    dados = predicoes_df[
        predicoes_df["modelo"] == nome_modelo
    ]

    fig, ax = plt.subplots(figsize=(6, 6))

    ax.scatter(
        dados["y_true"],
        dados["y_pred"],
        alpha=0.6,
    )

    min_val = min(
        dados["y_true"].min(),
        dados["y_pred"].min(),
    )

    max_val = max(
        dados["y_true"].max(),
        dados["y_pred"].max(),
    )

    ax.plot(
        [min_val, max_val],
        [min_val, max_val],
        linestyle="--",
    )

    ax.set_xlabel("LYM real")
    ax.set_ylabel("LYM predito")
    ax.set_title(
        f"{nome_modelo} — Real × Predito"
    )

    plt.show()